# SPINE-GPE v7 — PNADc Certifier v1.1.0

Certificação dos suplementos diretos de plataformas digitais da PNADc em **2022T4** e **2024T3**.

Regra central:

```python
platform_delivery_direct = (SD14001 == 1) & (S140093 == 1)
```

O notebook preserva `S140093 == 1` separadamente como uso bruto de aplicativo de entrega.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7")
ROOT.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("Existe:", ROOT.exists())

Mounted at /content/drive
ROOT: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7
Existe: True


In [3]:
# Localiza o script em /content ou na pasta persistente scripts/.
SCRIPT_NAME = "SPINE_GPEv7_PNADC_CERTIFIER_v1.1.0.py"
SCRIPT_CANDIDATES = [
    Path("/content") / SCRIPT_NAME,
    ROOT / "scripts" / SCRIPT_NAME,
    ROOT / SCRIPT_NAME,
]
SCRIPT = next((p for p in SCRIPT_CANDIDATES if p.exists()), None)

if SCRIPT is None:
    raise FileNotFoundError(
        "Envie SPINE_GPEv7_PNADC_CERTIFIER_v1.1.0.py para /content "
        "ou coloque-o em SPINE-GPEv7/scripts/."
    )
print("SCRIPT:", SCRIPT)

SCRIPT: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_PNADC_CERTIFIER_v1.1.0.py


In [4]:
# Instala somente dependências ausentes. Não depende de arquivo requirements.
import importlib
import subprocess
import sys

PACKAGES = {
    "numpy": "numpy>=1.26",
    "pandas": "pandas>=2.2",
    "pyarrow": "pyarrow>=17",
    "scipy": "scipy>=1.12",
    "requests": "requests>=2.31",
    "urllib3": "urllib3>=2.2",
    "bs4": "beautifulsoup4>=4.12",
    "openpyxl": "openpyxl>=3.1",
    "xlrd": "xlrd>=2.0",
    "lxml": "lxml>=5",
}

missing = []
for module, package in PACKAGES.items():
    try:
        imported = importlib.import_module(module)
        version = getattr(imported, "__version__", "disponível")
        print(f"OK       {module}: {version}")
    except ImportError:
        print(f"FALTANDO {module} -> {package}")
        missing.append(package)

if missing:
    result = subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-cache-dir", "--prefer-binary",
            "--retries", "10", "--timeout", "120",
            *missing,
        ],
        text=True,
        capture_output=True,
        check=False,
    )
    print(result.stdout[-12000:])
    print(result.stderr[-12000:])
    if result.returncode != 0:
        raise RuntimeError(f"pip falhou com exit code {result.returncode}")
else:
    print("Todas as dependências já estão disponíveis.")

OK       numpy: 2.0.2
OK       pandas: 2.2.2
OK       pyarrow: 18.1.0
OK       scipy: 1.16.3
OK       requests: 2.32.4
OK       urllib3: 2.5.0
OK       bs4: 4.13.5
OK       openpyxl: 3.1.5
OK       xlrd: 2.0.2
OK       lxml: 6.1.1
Todas as dependências já estão disponíveis.


In [5]:
# Validação sintática do script.
import py_compile
py_compile.compile(str(SCRIPT), doraise=True)
print("py_compile: OK")

py_compile: OK


In [6]:
# Confirma os microdados e o lock da Fase 0.
import json

DATA_PNADC = ROOT / "data_pnadc"
for expected in ("PNADC_042022.txt", "PNADC_032024.txt"):
    matches = list(DATA_PNADC.rglob(expected))
    print(expected, "->", matches[:3])

PHASE0_LOCK = ROOT / "00_admin" / "PHASE0_LOCK.json"
print("PHASE0_LOCK:", PHASE0_LOCK)
if not PHASE0_LOCK.exists():
    raise FileNotFoundError(PHASE0_LOCK)
phase0 = json.loads(PHASE0_LOCK.read_text(encoding="utf-8"))
print("Fase 0:", phase0.get("status"))
if phase0.get("status") != "RELEASED":
    raise RuntimeError("A Fase 0 precisa estar RELEASED.")

PNADC_042022.txt -> [PosixPath('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/data_pnadc/PNADC_042022.txt')]
PNADC_032024.txt -> [PosixPath('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/data_pnadc/PNADC_032024.txt')]
PHASE0_LOCK: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/PHASE0_LOCK.json
Fase 0: RELEASED


In [7]:
# Auditoria rápida: fontes, layouts, largura fixed-width e variáveis críticas.
audit_cmd = [
    sys.executable, str(SCRIPT),
    "--root", str(ROOT),
    "--mode", "audit",
    "--strict",
]
print(" ".join(audit_cmd))
audit = subprocess.run(audit_cmd, text=True, capture_output=True, check=False)
print(audit.stdout[-16000:])
print(audit.stderr[-16000:])
print("Audit exit code:", audit.returncode)
if audit.returncode != 0:
    raise RuntimeError("Auditoria bloqueada. Examine o log acima.")

/usr/bin/python3 /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_PNADC_CERTIFIER_v1.1.0.py --root /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 --mode audit --strict
2026-07-20 15:31:58,672 | INFO | SPINE-GPE PNADc Certifier v1.1.0 | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 | mode=audit
2026-07-20 15:32:23,163 | INFO | Microdado direto 2022 selecionado: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/10_ibge/platform_direct_supplements/2022q4/PNADC_2022_trimestre4.txt (largura=7362)
2026-07-20 15:33:19,026 | INFO | Microdado direto 2024 selecionado: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/10_ibge/platform_direct_supplements/2024q3/PNADC_2024_trimestre3.txt (largura=3846)
2026-07-20 15:33:37,756 | INFO | Auditoria concluída: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/reports/PNADC_CERTIFICATION_REPORT_20260720T153158Z.md | status=CERTIFIED


Audit exit code: 0


In [8]:
# Certificação completa. Não use --skip-sidra.
certify_cmd = [
    sys.executable, str(SCRIPT),
    "--root", str(ROOT),
    "--mode", "certify",
    "--chunk-rows", "20000",
    "--strict",
]
print(" ".join(certify_cmd))
certify = subprocess.run(certify_cmd, text=True, capture_output=True, check=False)
print(certify.stdout[-24000:])
print(certify.stderr[-24000:])
print("Certification exit code:", certify.returncode)

/usr/bin/python3 /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_PNADC_CERTIFIER_v1.1.0.py --root /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 --mode certify --chunk-rows 20000 --strict
2026-07-20 15:34:00,953 | INFO | SPINE-GPE PNADc Certifier v1.1.0 | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 | mode=certify
2026-07-20 15:34:05,604 | INFO | Microdado direto 2022 selecionado: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/10_ibge/platform_direct_supplements/2022q4/PNADC_2022_trimestre4.txt (largura=7362)
2026-07-20 15:34:36,586 | INFO | Microdado direto 2024 selecionado: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/10_ibge/platform_direct_supplements/2024q3/PNADC_2024_trimestre3.txt (largura=3846)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is n

In [9]:
# Abre o lock e o relatório mais recente.
LOCK = ROOT / "00_admin" / "PNADC_CERTIFICATION_LOCK.json"
if not LOCK.exists():
    raise FileNotFoundError(LOCK)
lock = json.loads(LOCK.read_text(encoding="utf-8"))
print(json.dumps(lock, ensure_ascii=False, indent=2)[:30000])

REPORT = ROOT / "06_reports" / "pnadc_certification" / "pnadc_certification_report_LATEST.md"
if REPORT.exists():
    report_text = REPORT.read_text(encoding="utf-8", errors="replace")
    print("\n--- RELATÓRIO FINAL ---\n")
    print(report_text[-30000:])

status = lock.get("status")
if status == "BLOCKED":
    raise RuntimeError("Certificação bloqueada; use critical_failures para a próxima correção.")
print("STATUS:", status)

{
  "run_id": "20260720T153400Z",
  "script_version": "1.1.0",
  "schema_version": "spine-gpe-v7-pnadc-certified-1.1.0",
  "status": "BLOCKED",
  "critical_failures": [
    {
      "test_id": "golden.2022.platform_any_total",
      "status": "BLOCKED",
      "severity": "critical",
      "year": 2022,
      "message": "Valor oficial SIDRA não foi selecionado de forma inequívoca.",
      "observed": 1319311.95782783,
      "expected": null,
      "tolerance": 0.015,
      "evidence": {
        "table": 9432,
        "reason": "no_match"
      }
    },
    {
      "test_id": "golden.2022.platform_any_percent",
      "status": "BLOCKED",
      "severity": "critical",
      "year": 2022,
      "message": "Valor oficial SIDRA não foi selecionado de forma inequívoca.",
      "observed": 1.5411028404767375,
      "expected": null,
      "tolerance": 0.02,
      "evidence": {
        "table": 9432,
        "reason": "no_match"
      }
    },
    {
      "test_id": "golden.2024.platform_any_tot

RuntimeError: Certificação bloqueada; use critical_failures para a próxima correção.

In [10]:
# Golden check independente da regra de entrega nos Parquets produzidos.
import pandas as pd

BASE = ROOT / "03_processed" / "10_pnadc_certified"
for year in (2022, 2024):
    path = BASE / f"certified_pnadc_platform_{year}.parquet"
    df = pd.read_parquet(
        path,
        columns=[
            "SD14001", "S140093", "delivery_app_use_raw",
            "platform_delivery_direct", "delivery_app_use_nonplatform",
            "survey_weight",
        ],
    )
    w = pd.to_numeric(df["survey_weight"], errors="coerce").fillna(0)
    raw = float(w[df["delivery_app_use_raw"].fillna(False)].sum())
    official = float(w[df["platform_delivery_direct"].fillna(False)].sum())
    nonplatform = float(w[df["delivery_app_use_nonplatform"].fillna(False)].sum())
    print(f"\n{year}")
    print(f"Uso bruto S140093=1: {raw:,.0f}")
    print(f"Entrega oficial SD14001=1 & S140093=1: {official:,.0f}")
    print(f"Uso não plataformizado: {nonplatform:,.0f}")
    print(f"Erro de partição: {raw - official - nonplatform:,.8f}")


2022
Uso bruto S140093=1: 570,639
Entrega oficial SD14001=1 & S140093=1: 445,867
Uso não plataformizado: 124,772
Erro de partição: 0.00000000

2024
Uso bruto S140093=1: 622,340
Entrega oficial SD14001=1 & S140093=1: 487,285
Uso não plataformizado: 135,055
Erro de partição: 0.00000000


In [11]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import unicodedata
import zipfile

import pandas as pd


ROOT = Path(
    "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7"
)

SNAPSHOT_DIR = (
    ROOT
    / "02_interim"
    / "10_pnadc_certification"
    / "sidra_snapshots"
)

STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

OUTPUT_DIR = (
    ROOT
    / "00_admin"
    / "reports"
    / f"SIDRA_SELECTOR_DIAGNOSTIC_{STAMP}"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


TABLES = {
    9432: {
        "key": "platform_any",
        "tokens": ("plataform",),
    },
    9441: {
        "key": "platform_delivery",
        "tokens": ("plataform", "entrega"),
    },
    9442: {
        "key": "income",
        "tokens": ("plataform", "rendimento"),
    },
    9443: {
        "key": "hours",
        "tokens": ("plataform", "hora"),
    },
    9642: {
        "key": "social_security",
        "tokens": ("plataform", "previd"),
    },
    9518: {
        "key": "informality",
        "tokens": ("plataform", "informal"),
    },
}


def normalize(value) -> str:
    text = "" if value is None else str(value)

    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char
        for char in text
        if not unicodedata.combining(char)
    )

    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()

    return text


def read_snapshot(path: Path) -> pd.DataFrame:
    encodings = (
        "utf-8-sig",
        "utf-8",
        "latin-1",
    )

    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(
                path,
                dtype="string",
                encoding=encoding,
                keep_default_na=False,
                low_memory=False,
            )
        except Exception as exc:
            last_error = exc

    raise RuntimeError(
        f"Não foi possível ler {path}: {last_error}"
    )


def latest_snapshot(table: int) -> Path | None:
    candidates = list(
        SNAPSHOT_DIR.glob(f"sidra_{table}_*.csv")
    )

    if not candidates:
        return None

    return max(
        candidates,
        key=lambda path: path.stat().st_mtime,
    )


def relevant_column(column: str) -> bool:
    name = normalize(column)

    terms = (
        "codigo",
        "variavel",
        "unidade",
        "ano",
        "periodo",
        "nivel territorial",
        "brasil",
        "plataform",
        "trabalh",
        "rendimento",
        "hora",
        "previd",
        "contribu",
        "informal",
        "tipo de plataforma",
    )

    return any(term in name for term in terms)


diagnostic = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "snapshot_directory": str(SNAPSHOT_DIR),
    "tables": {},
}


for table, specification in TABLES.items():
    snapshot = latest_snapshot(table)

    table_result = {
        "table": table,
        "key": specification["key"],
        "snapshot": (
            str(snapshot)
            if snapshot is not None
            else None
        ),
    }

    if snapshot is None:
        table_result["status"] = "SNAPSHOT_NOT_FOUND"
        diagnostic["tables"][str(table)] = table_result

        print(
            f"\nTabela {table}: snapshot não encontrado."
        )
        continue

    df = read_snapshot(snapshot)

    row_text = (
        df.fillna("")
        .astype(str)
        .agg(" | ".join, axis=1)
        .map(normalize)
    )

    year_mask = row_text.str.contains(
        r"\b2022\b|\b2024\b",
        regex=True,
        na=False,
    )

    brazil_mask = row_text.str.contains(
        r"\bbrasil\b",
        regex=True,
        na=False,
    )

    topic_mask = pd.Series(
        True,
        index=df.index,
        dtype=bool,
    )

    for token in specification["tokens"]:
        topic_mask &= row_text.str.contains(
            normalize(token),
            regex=False,
            na=False,
        )

    candidates = df.loc[
        year_mask & brazil_mask & topic_mask
    ].copy()

    # Caso o filtro temático seja excessivamente estrito,
    # conserva todas as linhas Brasil 2022/2024 para auditoria.
    fallback_used = False

    if candidates.empty:
        candidates = df.loc[
            year_mask & brazil_mask
        ].copy()

        fallback_used = True

    candidate_path = (
        OUTPUT_DIR
        / f"sidra_{table}_{specification['key']}_candidates.csv"
    )

    candidates.to_csv(
        candidate_path,
        index=False,
        encoding="utf-8-sig",
    )

    column_values = {}

    for column in df.columns:
        if not relevant_column(column):
            continue

        values = sorted(
            {
                str(value).strip()
                for value in candidates[column].tolist()
                if str(value).strip()
            }
        )

        column_values[column] = values[:200]

    table_result.update(
        {
            "status": "OK",
            "rows_snapshot": int(len(df)),
            "columns": list(df.columns),
            "rows_candidates": int(len(candidates)),
            "fallback_used": fallback_used,
            "candidate_path": str(candidate_path),
            "relevant_column_values": column_values,
        }
    )

    diagnostic["tables"][str(table)] = table_result

    print("\n" + "=" * 100)
    print(
        f"TABELA {table} — {specification['key']}"
    )
    print("Snapshot:", snapshot)
    print("Dimensão:", df.shape)
    print("Candidatos:", len(candidates))
    print("Fallback:", fallback_used)

    print("\nCOLUNAS:")
    for column in df.columns:
        print(" -", column)

    print("\nVALORES RELEVANTES:")
    for column, values in column_values.items():
        print(f"\n[{column}]")
        for value in values[:40]:
            print("  ", value)

    print("\nPRIMEIRAS LINHAS CANDIDATAS:")

    if candidates.empty:
        print("Nenhuma linha candidata.")
    else:
        with pd.option_context(
            "display.max_columns",
            None,
            "display.max_colwidth",
            180,
            "display.width",
            300,
            "display.max_rows",
            30,
        ):
            print(candidates.head(30).to_string(index=False))


summary_path = OUTPUT_DIR / "sidra_selector_diagnostic.json"

summary_path.write_text(
    json.dumps(
        diagnostic,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


zip_path = Path(
    f"/content/SIDRA_SELECTOR_DIAGNOSTIC_{STAMP}.zip"
)

with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            archive.write(
                path,
                arcname=path.relative_to(OUTPUT_DIR),
            )

    # Inclui os snapshots realmente utilizados.
    for table in TABLES:
        snapshot = latest_snapshot(table)

        if snapshot is not None:
            archive.write(
                snapshot,
                arcname=(
                    "source_snapshots"
                    f"/{snapshot.name}"
                ),
            )


print("\n" + "=" * 100)
print("Diagnóstico salvo em:", OUTPUT_DIR)
print("ZIP para upload:", zip_path)


TABELA 9432 — platform_any
Snapshot: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/02_interim/10_pnadc_certification/sidra_snapshots/sidra_9432_platform_any.csv
Dimensão: (24, 13)
Candidatos: 24
Fallback: True

COLUNAS:
 - Nível Territorial (Código)
 - Nível Territorial
 - Unidade de Medida (Código)
 - Unidade de Medida
 - Valor
 - Brasil (Código)
 - Brasil
 - Variável (Código)
 - Variável
 - Ano (Código)
 - Ano
 - Trabalho por meio de plataforma digital de serviço no trabalho principal (Código)
 - Trabalho por meio de plataforma digital de serviço no trabalho principal

VALORES RELEVANTES:

[Nível Territorial (Código)]
   1

[Nível Territorial]
   Brasil

[Unidade de Medida (Código)]
   1572
   2

[Unidade de Medida]
   %
   Mil pessoas

[Brasil (Código)]
   1

[Brasil]
   Brasil

[Variável (Código)]
   12900
   12901
   12902
   12903

[Variável]
   Coeficiente de variação - Distribuição percentual das pessoas de 14 anos ou mais de idade ocupadas na semana de referência, exc